# Tarea 4 · Paso 0 — Preparar los datos de lluvia para la cuenca de Imata
**MCHV-502 · Programación y SIG para el análisis de cuencas · CATIE**

- **Nombre:** Miguel Ángel Lázaro Nina
- **Área de interés:** microcuenca de cabecera del río Sumbay (Imata, Arequipa), 190,3 km², de 4489 a 5036 msnm (Tarea 3)

Este cuaderno produce los dos insumos de la Tarea 4:

1. `lluvia_promedio_2015_2025_imata.tif`: lluvia media anual 2015–2025 de CHIRPS v3, con el mismo formato que el ráster del profesor (celdas de 0,05°, EPSG:4326, mm/año, sin dato = −9999).
2. `estaciones_lluvia_imata.csv`: las estaciones del SENAMHI que miden lluvia a menos de 100 km de la cuenca, con el valor del ráster en cada una.

**Por qué no sirve el ráster del profesor.** Cubre de 0° a 35° de latitud norte (Centroamérica y el Caribe); Perú está en el hemisferio sur.

**Advertencia sobre los valores.** La lluvia de cada estación sale de CHIRPS, no de su pluviómetro. Es un valor *estimado*, anclado en observaciones, y así hay que declararlo.

## 1. Configuración

In [ ]:
!pip install -q rasterio

import os
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import from_bounds
import matplotlib.pyplot as plt

# --- Parámetros: lo único que cambiaría para otra cuenca ---
URL_ESTACIONES = "https://raw.githubusercontent.com/lazaroninamiguel-dot/mchv502-cuencas/refs/heads/main/estaciones_peru.csv"
RUTA_CUENCA    = "https://raw.githubusercontent.com/lazaroninamiguel-dot/mchv502-cuencas/refs/heads/main/T03_cuenca_imata_chili.geojson"

ANIOS     = list(range(2015, 2026))         # 2015 a 2025, igual que el ráster del profesor
BBOX      = (-72.5, -17.25, -69.5, -14.5)   # oeste, sur, este, norte: la cuenca + 100 km de margen
RADIO_KM  = 100                             # estaciones a menos de 100 km del borde de la cuenca
CRS_UTM   = "EPSG:32719"                    # UTM 19S: la zona de la cuenca, para medir en metros
NODATA    = -9999.0

SALIDA_TIF = "lluvia_promedio_2015_2025_imata.tif"
SALIDA_CSV = "estaciones_lluvia_imata.csv"

# Plantilla de las direcciones de CHIRPS v3 (un GeoTIFF mundial por año, en mm/año)
URL_CHIRPS = "https://data.chc.ucsb.edu/products/CHIRPS/v3.0/annual/global/tifs/chirps-v3.0.{anio}.tif"

## 2. Carga de datos

### 2.1 La cuenca de la Tarea 3
Antes de correr esta celda, suba `T03_cuenca_imata_chili.geojson` a su repositorio de GitHub (la misma carpeta de `estaciones_peru.csv`). Si prefiere, súbalo a Colab con el ícono de carpeta y cambie `RUTA_CUENCA` por `"T03_cuenca_imata_chili.geojson"`.

In [ ]:
cuenca = gpd.read_file(RUTA_CUENCA).to_crs("EPSG:4326")
area_km2 = cuenca.to_crs(CRS_UTM).area.sum() / 1e6     # el área se mide en metros, nunca en grados
print(f"Cuenca: {len(cuenca)} polígono(s), {area_km2:.1f} km2")
print("Extensión (oeste, sur, este, norte):", cuenca.total_bounds.round(4))

### 2.2 Lluvia anual de CHIRPS v3, año por año
Cada archivo de CHIRPS cubre todo el planeta. En vez de bajarlo entero, se lee solo la **ventana** (**window**) del recuadro `BBOX`: es como abrir solo la compuerta del canal que riega su parcela, no la represa completa. Si el servidor no permite leer por ventana, el archivo se descarga, se recorta y se borra.

In [ ]:
def leer_anio(anio):
    """Devuelve la lluvia del año recortada al BBOX (matriz) y el perfil del recorte."""
    url = URL_CHIRPS.format(anio=anio)
    try:
        fuente = "/vsicurl/" + url                      # lectura remota, solo la ventana
        with rasterio.open(fuente) as src:
            return _recortar(src)
    except Exception:
        os.system(f"wget -q -O chirps_tmp.tif {url}")    # plan B: descargar el año completo
        with rasterio.open("chirps_tmp.tif") as src:
            datos = _recortar(src)
        os.remove("chirps_tmp.tif")                       # se borra para no llenar el disco
        return datos

def _recortar(src):
    ventana = from_bounds(*BBOX, transform=src.transform).round_offsets().round_lengths()
    arr = src.read(1, window=ventana).astype("float32")
    arr[(arr == src.nodata) | (arr < 0)] = np.nan          # sin dato -> NaN, nunca -9999 en los cálculos
    perfil = src.profile.copy()
    perfil.update(height=arr.shape[0], width=arr.shape[1],
                  transform=src.window_transform(ventana))
    return arr, perfil

capas, anios_ok = [], []
for anio in ANIOS:
    try:
        arr, perfil = leer_anio(anio)
        capas.append(arr); anios_ok.append(anio)
        print(f"{anio}: {np.nanmean(arr):7.1f} mm de media en el recuadro")
    except Exception as e:
        print(f"{anio}: no disponible ({type(e).__name__})")

print(f"\nAños leídos: {len(anios_ok)} de {len(ANIOS)} -> {anios_ok}")

## 3. Procesamiento

### 3.1 Promedio 2015–2025 y exportación del GeoTIFF
El promedio se calcula celda por celda: cada celda es un pluviómetro virtual, y se promedian sus once años. `np.nanmean` ignora los vacíos en vez de tratarlos como cero.

In [ ]:
pila = np.stack(capas)                       # 3D: (años, filas, columnas)
media = np.nanmean(pila, axis=0)             # promedio a lo largo de los años

# Mismo formato que el ráster del profesor
perfil.update(driver="GTiff", dtype="float32", count=1, nodata=NODATA,
              crs="EPSG:4326", compress="deflate", tiled=True, blockxsize=256, blockysize=256)
salida = np.where(np.isnan(media), NODATA, media).astype("float32")

with rasterio.open(SALIDA_TIF, "w", **perfil) as dst:
    dst.write(salida, 1)
    dst.update_tags(fuente="CHIRPS v3.0 anual (Climate Hazards Center, UCSB)",
                    periodo=f"{anios_ok[0]}-{anios_ok[-1]}", unidades="mm/anio",
                    n_anios=str(len(anios_ok)))

with rasterio.open(SALIDA_TIF) as r:
    print(r.profile["width"], "x", r.profile["height"], "celdas de", r.res[0], "grados")
    print("Extensión:", r.bounds)
    print(f"Lluvia: {np.nanmin(media):.0f} a {np.nanmax(media):.0f} mm/año")

### 3.2 Estaciones que miden lluvia, filtradas por distancia
El filtro es por **distancia al borde de la cuenca**, no por departamento: la lluvia no respeta los límites políticos, y la cuenca está a 16 km de Puno. Las distancias se miden en UTM 19S (metros).

In [ ]:
est = pd.read_csv(URL_ESTACIONES, dtype={"codigo": str})     # codigo como texto: conserva 000765
est = est[est["mide_lluvia"] == "Sí"]                        # fuera las hidrológicas
gest = gpd.GeoDataFrame(est, geometry=gpd.points_from_xy(est["longitud"], est["latitud"]),
                        crs="EPSG:4326")

# Distancia de cada estación al borde de la cuenca, en km
cuenca_utm = cuenca.to_crs(CRS_UTM)
borde = cuenca_utm.union_all() if hasattr(cuenca_utm, "union_all") else cuenca_utm.unary_union
gest["dist_km"] = gest.to_crs(CRS_UTM).distance(borde) / 1000
cerca = gest[gest["dist_km"] <= RADIO_KM].copy()

cerca = cerca.sort_values("dist_km")
print(f"Estaciones de lluvia a menos de {RADIO_KM} km: {len(cerca)}")
print(cerca.groupby("subregion").size())

### 3.3 Valor del ráster en cada estación
`sample` lee el valor de la celda donde cae cada punto, como si se pusiera un pluviómetro justo ahí. Luego se deja **una estación por celda** de 0,05° (unos 5,5 km).

In [ ]:
with rasterio.open(SALIDA_TIF) as r:
    coords = list(zip(cerca["longitud"], cerca["latitud"]))      # (x, y) = (longitud, latitud)
    cerca["lluvia_mm"] = [v[0] for v in r.sample(coords)]
    cerca["celda"] = [r.index(x, y) for x, y in coords]           # (fila, columna) de cada estación

# Una estación por celda. Dos estaciones en la misma celda (p. ej. la convencional y la
# automática de Imata) reciben el MISMO valor: en la validación cruzada una "adivinaría"
# perfecto a la otra y el error saldría falsamente bajo. Se deja la convencional.
orden_tipo = {"CO": 0, "CP": 0, "MAP": 1, "PLU": 1, "EMA": 2, "EAMA": 2, "EHMA": 3, "EAA": 3}
cerca["prioridad"] = cerca["tipo"].map(orden_tipo).fillna(9)
antes = len(cerca)
cerca = (cerca.sort_values(["prioridad", "dist_km"])
              .drop_duplicates(subset="celda")
              .drop(columns=["prioridad", "celda"])
              .sort_values("dist_km"))
print(f"Una por celda: {antes} -> {len(cerca)} estaciones")

cerca.loc[cerca["lluvia_mm"] == NODATA, "lluvia_mm"] = np.nan
sin_valor = cerca["lluvia_mm"].isna().sum()
cerca = cerca.dropna(subset=["lluvia_mm"])
cerca["lluvia_mm"] = cerca["lluvia_mm"].round(1)
print(f"Estaciones con valor: {len(cerca)} (sin valor: {sin_valor})")
print(f"Lluvia en las estaciones: {cerca['lluvia_mm'].min()} a {cerca['lluvia_mm'].max()} mm/año")

columnas = ["codigo", "nombre", "tipo", "subregion", "latitud", "longitud",
            "altitud_msnm", "dist_km", "lluvia_mm"]
tabla = cerca[columnas].round({"dist_km": 1})
tabla.to_csv(SALIDA_CSV, index=False)
tabla.head(10)

## 4. Visualización

In [ ]:
from matplotlib.lines import Line2D
fig, ax = plt.subplots(figsize=(9, 8))

with rasterio.open(SALIDA_TIF) as r:
    lluvia = r.read(1, masked=True)
    ext = [r.bounds.left, r.bounds.right, r.bounds.bottom, r.bounds.top]
img = ax.imshow(lluvia, extent=ext, cmap="Blues", origin="upper")
plt.colorbar(img, ax=ax, label="Lluvia media anual 2015–2025 (mm)", shrink=0.8)

# Radio de búsqueda: la cuenca ampliada 100 km (se dibuja en grados)
cuenca.to_crs(CRS_UTM).buffer(RADIO_KM * 1000).to_crs("EPSG:4326").boundary.plot(
    ax=ax, color="gray", linestyle="--", linewidth=1)
cuenca.boundary.plot(ax=ax, color="red", linewidth=2)
gpd.GeoDataFrame(tabla, geometry=gpd.points_from_xy(tabla.longitud, tabla.latitud),
                 crs="EPSG:4326").plot(ax=ax, color="black", markersize=25, marker="^")

ax.legend(handles=[Line2D([], [], color="red", lw=2, label="Cuenca de Imata (Tarea 3)"),
                   Line2D([], [], color="gray", ls="--", label=f"Radio de {RADIO_KM} km"),
                   Line2D([], [], color="black", marker="^", ls="", label=f"Estaciones ({len(tabla)})")],
          loc="lower left")
ax.set_title("Lluvia media anual CHIRPS v3 (2015–2025) y estaciones del SENAMHI")
ax.set_xlabel("Longitud (°)"); ax.set_ylabel("Latitud (°)")
plt.show()

## 5. Resumen

In [ ]:
with rasterio.open(SALIDA_TIF) as r:
    from rasterio.mask import mask
    rec, _ = mask(r, cuenca.geometry, crop=True, nodata=NODATA)
vals = rec[0][rec[0] != NODATA]

print("ANIOS_CHIRPS           =", f"{anios_ok[0]}-{anios_ok[-1]} ({len(anios_ok)} años)")
print("N_ESTACIONES           =", len(tabla))
print("LLUVIA_ESTACIONES_MM   =", (float(tabla.lluvia_mm.min()), float(tabla.lluvia_mm.max())))
print("CELDAS_CUENCA_RASTER   =", vals.size)
print("LLUVIA_CUENCA_RASTER   =", round(float(vals.mean()), 1), "mm/año (referencia para validar la Tarea 4)")
print("Archivos:", SALIDA_TIF, "|", SALIDA_CSV)

**Siguiente paso.** Suba `lluvia_promedio_2015_2025_imata.tif` y `estaciones_lluvia_imata.csv` a su repositorio de GitHub. En la Tarea 4 se interpolan las estaciones (vecino más cercano e IDW) y la lluvia media obtenida se compara con `LLUVIA_CUENCA_RASTER`.

**Dato para la interpretación.** Imata registró 537 mm/año en 1962–2012 (SENAMHI). Compárelo con el valor que CHIRPS le asigna en 2015–2025: son periodos distintos, así que la diferencia no es solo error del producto.